In [1]:
from google.colab import drive
import os
drive.mount('/content/drive/', force_remount=True)

os.chdir('drive/MyDrive/ExoVision')

Mounted at /content/drive/


In [ ]:
pip install -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.7/133.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 27.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 63.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 35.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.3/491.3 kB 33.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 39.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 35.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 31.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 MB 9.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 45.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.5 MB/s eta 0:0

In [2]:
import sys
import os
import time
from lightkurve import search_targetpixelfile
from lightkurve import TessTargetPixelFile
import lightkurve as lk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from keras.models import load_model
from keras.optimizers import Adam
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Conv1D, MaxPooling1D, Flatten
from wotan import slide_clip
from wotan import transit_mask, flatten
from astropy.stats import sigma_clip
from astropy import units as u
import csv
import shutil
from scipy.interpolate import interp1d
from tsfresh import extract_features
from tsfresh.utilities.dataframe_functions import make_forecasting_frame
from tsfresh.utilities.dataframe_functions import impute
from statsmodels.tsa.seasonal import seasonal_decompose
from multiprocessing import Pool
import multiprocessing
import numpy as np
import pandas as pd
import lightkurve as lk
from scipy.signal import find_peaks
from astropy.timeseries import BoxLeastSquares
from scipy.interpolate import interp1d

In [3]:
%matplotlib inline

In [4]:
value_df = 10000
cwd = os.getcwd()
dirname = cwd[len(cwd)-len("Satellite Datasets"):len(cwd)]
if(dirname != 'Satellite Datasets'):
    os.chdir('./Satellite Datasets')
datasets = os.listdir()
stars1 = pd.read_csv('star1.csv')
stars2 = pd.read_csv('star2.csv')
stars3 = pd.read_csv('star3.csv')
stars4 = pd.read_csv('star4.csv')
stars = pd.concat([stars1,stars2,stars3,stars4])
stars = stars.drop_duplicates('target_name')
stars['target_name'] = stars['target_name'].apply(lambda x: 'TIC ' + str(x))
confirmed_stars = pd.read_csv('confirmed_stars.csv')
confirmed_stars['tid'] = confirmed_stars['tid'].apply(lambda x: 'TIC ' + str(x))
stars['confirmed_planet'] = stars['target_name'].isin(confirmed_stars['tid']).astype(int)
stars = stars.reset_index()
confirmed_index = np.array(stars[stars['confirmed_planet']==1].index)
unconfirmed_index = np.array(stars[stars['confirmed_planet']==0].index)

<ipython-input-4-2d99a23ce6f8>:8: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  stars2 = pd.read_csv('star2.csv')
<ipython-input-4-2d99a23ce6f8>:10: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  stars4 = pd.read_csv('star4.csv')


In [ ]:
# with open('exoplanet_star_updated_flux.csv', 'w', newline="", encoding="utf-8") as file:
#     csvwriter = csv.writer(file)
#     csvwriter.writerow(np.concatenate([['tid', 'confirmed_planet'], np.arange(1, 10001)]))

In [5]:
index_num = 2393
for planet_index in range(1660,7000):
    print('Starting index : ', planet_index)
    for conf_index in ['confirmed','nonconfirmed']:
        idno = ""
        planet = ""
        if conf_index == 'confirmed':
            idno = stars.iloc[confirmed_index[planet_index]]['target_name']
            planet = stars.iloc[confirmed_index[planet_index]]['confirmed_planet']
        else:
            idno = stars.iloc[unconfirmed_index[planet_index]]['target_name']
            planet = stars.iloc[unconfirmed_index[planet_index]]['confirmed_planet']
        sectorsdata = lk.search_lightcurve(idno, author=["TESS-SPOC"], exptime=1800)
        if(sectorsdata.download_all()!= None):
            sectors = sectorsdata.download_all()
            for i in sectors:
                i.flux = i.pdcsap_flux.value.unmasked
                i.flux_err = i.pdcsap_flux_err.value.unmasked
            stitched_lc = sectors.stitch()
            time = stitched_lc.time.value
            flux = stitched_lc.flux.value
            min_period = 1
            max_period = 1000
            num_periods = 10000
            period_time = np.logspace(np.log10(min_period), np.log10(max_period), num_periods)
            bls_periodogram = stitched_lc.to_periodogram(method='bls', period=period_time)
            planet_period = bls_periodogram.period_at_max_power
            planet_t0 = bls_periodogram.transit_time_at_max_power
            planet_duration = bls_periodogram.duration_at_max_power
            folded_light_curve = stitched_lc.fold(period=planet_period, epoch_time=planet_t0)
            flatten_lc, trend_lc = flatten(folded_light_curve.time.value, folded_light_curve.flux, method='biweight', return_trend=True)
            light = pd.DataFrame({'Time':folded_light_curve.time.value,'Flux':flatten_lc}).dropna()
            flux_series = pd.Series([i[0] for i in light[['Flux']].to_numpy()], index=[i[0] for i in light[['Time']].to_numpy()])
            decompose_result_mult = seasonal_decompose(flux_series, model="additive", period=int(planet_period.value))
            trend = decompose_result_mult.trend
            if trend.shape[0] < 10000:
                arr_val = np.full(value_df, pd.DataFrame(trend).mean()[0])
                arr_val[5000-(trend.shape[0]//2):5000+(trend.shape[0]//2)] = trend.to_numpy()[:2*(trend.shape[0]//2)]
            else:
                arr_val = trend.to_numpy()[(trend.shape[0]//2-5000):(trend.shape[0]//2+5000)]
            with open('exoplanet_star_updated_flux.csv', 'a', newline="", encoding="utf-8") as file:
                csvwriter = csv.writer(file)
                st = np.concatenate([[idno,planet],arr_val])
                csvwriter.writerow(st)
            index_num = index_num + 1
            print("Planets Added to the database",index_num)
    print("Succesfully Finished Adding Planet Index",planet_index,)

Starting index :  1456


`period` contains 254643 points.Periodogram is likely to be large, and slow to evaluate. Consider setting `frequency_factor` to a higher value.


Planets Added to the database 2100


<ipython-input-5-1a9585f90aff>:37: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  arr_val = np.full(value_df, pd.DataFrame(trend).mean()[0])


Planets Added to the database 2101
Succesfully Finished Adding Planet Index 1456
Starting index :  1457


...

`period` contains 247179 points.Periodogram is likely to be large, and slow to evaluate. Consider setting `frequency_factor` to a higher value.


Planets Added to the database 2392
Succesfully Finished Adding Planet Index 1658
Starting index :  1659


/usr/local/lib/python3.10/dist-packages/lightkurve/search.py:495: LightkurveWarning: Cannot download from an empty search result.
  warnings.warn(
`period` contains 247179 points.Periodogram is likely to be large, and slow to evaluate. Consider setting `frequency_factor` to a higher value.


Planets Added to the database 2393
Succesfully Finished Adding Planet Index 1659
Starting index :  1660


KeyboardInterrupt: 